# 🌐 Hugging Face Ecosystem: Complete Practical Masterclass

This notebook provides a complete hands-on walkthrough of the **Hugging Face Ecosystem**:

1. **Authentication & Token Management**: Secure token loading from `.env`, CLI/Python login, identity inspection.
2. **Hugging Face Hub Management**: Creating and managing repositories programmatically with `HfApi`.
3. **The `datasets` Library**: Ingestion, slicing, filtering, batched transformations (`.map`), and streaming.
4. **The `transformers` Library**: AutoClasses (`AutoTokenizer`, `AutoModel`), encoding, and decoding.
5. **High-Level `pipeline` Inference**: Sentiment analysis, Zero-shot classification, NER, and QA.
6. **Publishing Models to the Hub**: Namespace formatting, uploading models, tokenizers, and model cards.


## 1. Authentication & Token Management
Tokens are loaded securely from `.env` without exposing sensitive credentials in code.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Load secrets from .env (checking current directory and parent paths)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(".env")
load_dotenv("../.env")
load_dotenv("SLM_Experiment/.env")

READ_TOKEN = os.environ.get("HUGGINGFACE_READ_TOKEN")
WRITE_TOKEN = os.environ.get("HUGGINGFACE_WRITE_TOKEN")
FULL_ACCESS_TOKEN = os.environ.get("HUGGINGFACE_FULL_ACCESS_TOKEN")

print("Environment Tokens Status:")
print(f"- READ_TOKEN:        {'Loaded' if READ_TOKEN else 'Not Found'}")
print(f"- WRITE_TOKEN:       {'Loaded' if WRITE_TOKEN else 'Not Found'}")
print(f"- FULL_ACCESS_TOKEN: {'Loaded' if FULL_ACCESS_TOKEN else 'Not Found'}")


In [ ]:
from huggingface_hub import login, HfApi

# Select the token with write / full-access capabilities
active_token = FULL_ACCESS_TOKEN or WRITE_TOKEN or READ_TOKEN

if active_token:
    login(token=active_token, add_to_git_credential=True)
    print("Successfully authenticated with Hugging Face Hub!")
else:
    print("Warning: No token found. Running in unauthenticated read-only mode.")

# Verify user identity
api = HfApi()
user_info = api.whoami()
username = user_info.get("name")
print(f"Logged in User:   {username}")
print(f"Account Type:     {user_info.get('type')}")
print(f"Token Role:       {user_info.get('auth', {}).get('accessToken', {}).get('role', 'standard')}")


## 2. Hugging Face Hub & Repository Management
Hugging Face repos follow the `{username}/{repo_name}` convention.

In [ ]:
from huggingface_hub import create_repo, HfApi

# Define repository identifier
demo_repo_name = "huggingface-guide-demo"
demo_repo_id = f"{username}/{demo_repo_name}"

print(f"Target Repository ID: {demo_repo_id}")

# Create repo programmatically (exist_ok=True avoids error if already created)
try:
    repo_url = create_repo(
        repo_id=demo_repo_id,
        repo_type="model",
        private=True,          # Keep private for demonstration
        exist_ok=True,
        token=active_token
    )
    print(f"Repository confirmed at: {repo_url}")
except Exception as e:
    print(f"Repo creation note: {e}")


## 3. The `datasets` Library Masterclass
Loading, inspecting, filtering, mapping, and streaming datasets.

In [ ]:
from datasets import load_dataset

# 1. Download and load the standard IMDB movie reviews dataset
dataset = load_dataset("stanfordnlp/imdb")
print("Dataset Structure:")
print(dataset)

# Inspect features and column types
print("\nTrain Features:", dataset['train'].features)
print("Total Train Samples:", len(dataset['train']))
print("Total Test Samples: ", len(dataset['test']))


In [ ]:
# 2. Inspect an individual sample
sample = dataset['train'][0]
print("Sample Text Preview:", sample['text'][:150], "...")
print("Sample Label:       ", sample['label'], "('0'=Negative, '1'=Positive)")


In [ ]:
# 3. Dataset Transformations: Slicing, Filtering, and Mapping

# A. Slicing subsets (e.g. 500 samples for quick validation)
subset_train = dataset['train'].select(range(500))
print(f"Subset sample count: {len(subset_train)}")

# B. Filtering: Select only concise reviews (< 50 words)
short_reviews = dataset['train'].filter(lambda row: len(row['text'].split()) < 50)
print(f"Concise reviews count: {len(short_reviews)}")

# C. Batched Mapping: Fast parallel transformations
def add_word_count(batch):
    return {"word_count": [len(t.split()) for t in batch["text"]]}

subset_mapped = subset_train.map(add_word_count, batched=True, batch_size=100)
print("Sample with new feature:", subset_mapped[0]['word_count'], "words")


In [ ]:
# 4. Streaming Mode (Zero RAM Overhead for Massive Datasets)
print("Connecting to streaming dataset (no download necessary)...")
stream_ds = load_dataset("stanfordnlp/imdb", split="train", streaming=True)

# Fetch first 2 examples on-the-fly
print("\nFetched from stream:")
for i, example in enumerate(stream_ds.take(2)):
    print(f"[{i+1}] {example['text'][:100]}...")


## 4. The `transformers` Library: AutoClasses & Tokenizers
Using `AutoTokenizer` and `AutoModel` to dynamically handle any architecture.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_checkpoint = "bert-base-uncased"

# Load tokenizer and model using AutoClasses
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

print(f"Tokenizer Vocab Size: {tokenizer.vocab_size:,}")
print(f"Model Type:           {type(model).__name__}")


In [ ]:
# Encoding raw text into tensors
text = "Hugging Face makes natural language processing accessible and powerful."

inputs = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Input IDs:      ", inputs['input_ids'][0])
print("Attention Mask: ", inputs['attention_mask'][0])
print("Tokenized Subwords:", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
print("Decoded Text:   ", tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True))


## 5. The `pipeline` Abstraction: Zero-Code Inference
End-to-end inference pipelines for standard NLP tasks.

In [ ]:
from transformers import pipeline

# 1. Sentiment Analysis Pipeline
sentiment_pipe = pipeline("sentiment-analysis")
reviews = [
    "The performance was breathtaking and emotionally moving.",
    "Boring plot, wooden acting, and terrible sound design."
]

for r in reviews:
    res = sentiment_pipe(r)[0]
    print(f"Text: '{r}'")
    print(f"-> Label: {res['label']}, Confidence: {res['score']:.4f}\n")


In [ ]:
# 2. Zero-Shot Classification Pipeline (Classify without training on target labels)
zero_shot_pipe = pipeline("zero-shot-classification")

headline = "NVIDIA announced new Blackwell GPU architectures for generative AI models."
candidate_labels = ["technology", "sports", "politics", "healthcare"]

result = zero_shot_pipe(headline, candidate_labels=candidate_labels)
print(f"Headline: '{headline}'")
print(f"Top Predicted Category: {result['labels'][0]} ({result['scores'][0]*100:.2f}%)")


In [ ]:
# 3. Question Answering Pipeline
qa_pipe = pipeline("question-answering")

context = "Hugging Face was founded in 2016 by Clement Delangue, Julien Chaumond, and Thomas Wolf in New York City."
question = "When was Hugging Face founded?"

qa_res = qa_pipe(question=question, context=context)
print(f"Question: '{question}'")
print(f"Answer:   '{qa_res['answer']}' (Score: {qa_res['score']:.4f})")


## 6. Publishing Models & Artifacts to the Hub
Demonstration of pushing models and tokenizers to Hugging Face Hub.

In [ ]:
# Demonstration of publishing workflow
if active_token and username:
    target_repo = f"{username}/demo-bert-tokenizer"
    print(f"Target repository: https://huggingface.co/{target_repo}")
    
    # Create repo if not already created
    create_repo(repo_id=target_repo, exist_ok=True, token=active_token)
    
    # Push tokenizer to Hub
    # tokenizer.push_to_hub(target_repo, token=active_token)
    print(f"Ready to publish model and tokenizer to: {target_repo}")
else:
    print("Authentication required to push to Hub.")


## 7. Summary Reference

| Component | Primary Function | Key Classes / Functions |
| :--- | :--- | :--- |
| **Authentication** | Secure credentials and identity | `login()`, `HfApi()`, `api.whoami()` |
| **Hub Repos** | Host models, datasets, spaces | `create_repo()`, `upload_file()`, `upload_folder()` |
| **Datasets** | Data streaming, caching, mapping | `load_dataset()`, `.filter()`, `.map()`, `.set_format()` |
| **Transformers** | Architectures, weights, tokenizers | `AutoTokenizer`, `AutoModel`, `Trainer` |
| **Pipelines** | Zero-code task inference | `pipeline("sentiment-analysis")`, `pipeline("zero-shot-classification")` |
| **Sharing** | Uploading trained models to world | `model.push_to_hub()`, `tokenizer.push_to_hub()` |
